<a href="https://colab.research.google.com/github/njwbilll/Tugas-2_scikit-learn-Cookbook-O-Reilly-_Najwa-Bilqis-Al-Khalidah/blob/main/02_Pre_Model_Workflow_and_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 2: Pre-Model Workflow and Data Preprocessing

**Referensi:** scikit-learn Cookbook, Third Edition - John Sukup (Packt Publishing, 2025)

---

## Ringkasan Chapter

Chapter 2 membahas teknik-teknik preprocessing data yang esensial menggunakan transformers dan pipelines scikit-learn. Kualitas data adalah faktor terpenting dalam mencapai hasil yang baik dari model ML. Idiom "garbage in, garbage out" sangat relevan di sini: data yang buruk akan menghasilkan model yang buruk, tidak peduli seberapa canggih algoritma yang digunakan.

### Topik yang Dibahas:
1. Dampak raw data terhadap performa model
2. Menangani missing data (SimpleImputer, KNNImputer, IterativeImputer)
3. Teknik scaling (StandardScaler, MinMaxScaler, Normalizer)
4. Encoding variabel kategorikal (OneHotEncoder, LabelEncoder, ColumnTransformer)
5. Introduction to Pipelines dalam scikit-learn
6. Feature engineering (PolynomialFeatures, KBinsDiscretizer, RFE, SelectFromModel)


---
## Setup: Import Library

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Pengaturan tampilan pandas
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 10)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

print("Library berhasil diimport.")


Library berhasil diimport.


---
## 1. Dampak Raw Data terhadap Performa Model

### Penjelasan Teori

Algoritma ML dirancang untuk mempelajari pola dari data. Namun, ketika data input memiliki masalah seperti missing values, outlier, atau fitur yang tidak relevan, kemampuan model untuk melakukan generalisasi dari data training ke data yang belum dilihat akan berkurang.

**Masalah data yang paling umum:**

1. **Missing Data (Data yang Hilang):** Dataset yang tidak lengkap sangat umum dalam skenario dunia nyata. Missing data dapat muncul dari berbagai sumber seperti kesalahan pengumpulan data, kegagalan sistem, atau korupsi data.

2. **Outlier:** Nilai ekstrem yang dapat mendistorsi analisis statistik dan menghasilkan hasil yang menyesatkan. Outlier perlu diidentifikasi dan ditangani, misalnya dengan `RobustScaler()`.

3. **Variabel Kategorikal:** Sebagian besar algoritma ML memerlukan input numerik, sehingga variabel kategorikal perlu dikonversi, misalnya menggunakan `OneHotEncoder()` atau `LabelEncoder()`.

4. **Feature Scaling:** Fitur dengan skala berbeda dapat mempengaruhi konvergensi dan performa model, terutama untuk algoritma yang mengandalkan metrik jarak (seperti KNN). `StandardScaler()` dan `MinMaxScaler()` adalah teknik yang umum digunakan.

5. **Data Leakage:** Terjadi ketika informasi dari luar dataset training digunakan untuk membuat model, menghasilkan metrik performa yang terlalu optimis.


---
## 2. Menangani Missing Data

### Penjelasan Teori

Missing data harus ditangani sebelum melatih model ML karena sebagian besar algoritma tidak dapat menanganinya secara langsung. scikit-learn menyediakan tiga strategi utama untuk imputasi (pengisian) missing values:

1. **`SimpleImputer()`:** Mengganti nilai yang hilang dengan statistik sederhana seperti mean, median, atau modus. Paling cocok untuk dataset dengan nilai yang hilang kurang dari 5%.

2. **`KNNImputer()`:** Menggunakan algoritma K-Nearest Neighbors untuk mengimputasi nilai yang hilang berdasarkan nilai tetangga terdekat. Ideal untuk dataset dengan tingkat missing values 5-10% dan semua fitur numerik.

3. **`IterativeImputer()`:** Pendekatan canggih yang memodelkan setiap fitur dengan missing values sebagai fungsi dari fitur lainnya secara round-robin. Cocok untuk dataset campuran dengan tingkat missing values moderat hingga tinggi.

**Faktor yang menentukan pilihan strategi imputasi:**
- Tipe data (numerik vs kategorikal)
- Distribusi data
- Pola missing data (apakah hilang secara acak atau tidak)
- Persentase missing data dalam dataset

> **Catatan:** Tidak semua algoritma memerlukan imputasi. Decision Tree dan Random Forest dapat mengakomodasi fitur dengan missing values secara otomatis.


In [2]:
# Membuat dataset toy dengan missing values
np.random.seed(2024)
n_samples = 20
n_features = 10

data = {
    f"Feature{i+1}": np.random.uniform(0, 100, n_samples)
    for i in range(n_features)
}

df = pd.DataFrame(data)

# Menyisipkan missing values secara acak (~20% dari data)
for column in df.columns:
    mask = np.random.random(n_samples) < 0.2
    df.loc[mask, column] = np.nan

print(f"Shape dataset: {df.shape}")
print(f"Jumlah missing values per kolom:")
print(df.isnull().sum())
print()
print("5 baris pertama dataset:")
display(df.head())


Shape dataset: (20, 10)
Jumlah missing values per kolom:
Feature1     3
Feature2     5
Feature3     4
Feature4     4
Feature5     5
Feature6     5
Feature7     7
Feature8     6
Feature9     2
Feature10    3
dtype: int64

5 baris pertama dataset:


,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.8015,NaN,42.0098,34.6804,49.9623,NaN,NaN,89.9546,41.1524,NaN
1,69.9109,9.5542,6.4364,31.2878,37.9665,NaN,82.0056,NaN,75.7976,91.3825
2,NaN,96.0910,59.6433,84.7104,NaN,84.1090,NaN,70.2881,1.7783,4.1177
3,NaN,25.1767,83.7324,88.0231,16.8869,97.2056,84.6968,NaN,NaN,80.0780
4,20.5019,NaN,89.2486,67.6559,58.6359,78.2257,60.9116,NaN,65.1142,99.1192


### 2.1 SimpleImputer()

`SimpleImputer()` adalah metode paling sederhana dan ringan secara komputasi. Strategi yang tersedia:
- `"mean"`: Mengganti dengan nilai rata-rata fitur (untuk data numerik).
- `"median"`: Mengganti dengan nilai median fitur (lebih robust terhadap outlier).
- `"most_frequent"`: Mengganti dengan nilai yang paling sering muncul (untuk data kategorikal atau numerik).
- `"constant"`: Mengganti dengan nilai konstan yang ditentukan.


In [3]:
from sklearn.impute import SimpleImputer

# Inisialisasi SimpleImputer dengan strategi mean
imputer = SimpleImputer(strategy="mean")

# Fit dan transform data
imputed_data = imputer.fit_transform(df)
imputed_df = pd.DataFrame(imputed_data, columns=df.columns)

print("Dataset setelah SimpleImputer (strategy='mean'):")
display(imputed_df.head())
print(f"Missing values setelah imputasi: {imputed_df.isnull().sum().sum()}")


Dataset setelah SimpleImputer (strategy='mean'):


,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.8015,53.8647,42.0098,34.6804,49.9623,57.8230,51.1450,89.9546,41.1524,48.4000
1,69.9109,9.5542,6.4364,31.2878,37.9665,57.8230,82.0056,50.7150,75.7976,91.3825
2,52.5586,96.0910,59.6433,84.7104,46.0559,84.1090,51.1450,70.2881,1.7783,4.1177
3,52.5586,25.1767,83.7324,88.0231,16.8869,97.2056,84.6968,50.7150,60.6167,80.0780
4,20.5019,53.8647,89.2486,67.6559,58.6359,78.2257,60.9116,50.7150,65.1142,99.1192


Missing values setelah imputasi: 0


### 2.2 KNNImputer()

`KNNImputer()` mengimputasi missing values berdasarkan nilai tetangga terdekat dalam ruang fitur. Untuk setiap nilai yang hilang, KNNImputer mengidentifikasi K sampel terdekat (berdasarkan fitur yang tidak hilang) dan menggunakan rata-rata tertimbang jarak dari nilai K tetangga tersebut sebagai imputasi.

Parameter utama:
- `n_neighbors`: Jumlah tetangga terdekat yang digunakan (default: 5).


In [4]:
from sklearn.impute import KNNImputer

# Inisialisasi KNNImputer
knn_imputer = KNNImputer(n_neighbors=2)

# Fit dan transform data
knn_imputed_data = knn_imputer.fit_transform(df)
knn_imputed_df = pd.DataFrame(knn_imputed_data, columns=df.columns)

print("Dataset setelah KNNImputer (n_neighbors=2):")
display(knn_imputed_df.head())
print(f"Missing values setelah imputasi: {knn_imputed_df.isnull().sum().sum()}")


Dataset setelah KNNImputer (n_neighbors=2):


,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.8015,48.9540,42.0098,34.6804,49.9623,93.9104,52.5493,89.9546,41.1524,59.8337
1,69.9109,9.5542,6.4364,31.2878,37.9665,86.7935,82.0056,45.4162,75.7976,91.3825
2,67.7523,96.0910,59.6433,84.7104,73.3383,84.1090,59.0129,70.2881,1.7783,4.1177
3,47.8809,25.1767,83.7324,88.0231,16.8869,97.2056,84.6968,64.5103,39.7268,80.0780
4,20.5019,31.6709,89.2486,67.6559,58.6359,78.2257,60.9116,25.9036,65.1142,99.1192


Missing values setelah imputasi: 0


### 2.3 IterativeImputer()

`IterativeImputer()` adalah teknik imputasi yang canggih yang memodelkan setiap fitur dengan nilai hilang sebagai fungsi regresi, di mana fitur tersebut berperan sebagai output (y) dan semua fitur lain tanpa nilai hilang berperan sebagai input (X). Proses ini diulangi untuk setiap fitur dengan nilai hilang secara round-robin.

> **Catatan:** `IterativeImputer()` adalah fitur eksperimental dan memerlukan import khusus dengan `enable_iterative_imputer`.


In [5]:
from sklearn.experimental import enable_iterative_imputer  # Harus diimport untuk mengaktifkan
from sklearn.impute import IterativeImputer

# Inisialisasi IterativeImputer
iterative_imputer = IterativeImputer(random_state=42)

# Fit dan transform data
iterative_imputed_data = iterative_imputer.fit_transform(df)
iterative_imputed_df = pd.DataFrame(iterative_imputed_data, columns=df.columns)

print("Dataset setelah IterativeImputer:")
display(iterative_imputed_df.head())
print(f"Missing values setelah imputasi: {iterative_imputed_df.isnull().sum().sum()}")


Dataset setelah IterativeImputer:


,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.8015,54.3650,42.0098,34.6804,49.9623,58.6806,51.1781,89.9546,41.1524,48.2923
1,69.9109,9.5542,6.4364,31.2878,37.9665,60.0706,82.0056,50.7103,75.7976,91.3825
2,52.5397,96.0910,59.6433,84.7104,46.1213,84.1090,51.1554,70.2881,1.7783,4.1177
3,52.6306,25.1767,83.7324,88.0231,16.8869,97.2056,84.6968,50.6803,50.6509,80.0780
4,20.5019,53.1990,89.2486,67.6559,58.6359,78.2257,60.9116,50.6917,65.1142,99.1192


Missing values setelah imputasi: 0


---
## 3. Teknik Scaling

### Penjelasan Teori

Ketika bekerja dengan dataset, fitur-fitur dapat memiliki skala yang sangat berbeda. Banyak algoritma ML sensitif terhadap perbedaan skala ini. Scaling membantu memastikan tidak ada fitur tunggal yang mendominasi proses pembelajaran.

**Dua konsep penting yang sering membingungkan:**

- **Standardisasi (Standardization):** Mengubah data sehingga memiliki mean = 0 dan standar deviasi = 1. Menghasilkan Z-score. Formula: `z = (x - mu) / sigma`

- **Normalisasi (Normalization):** Mengubah rentang distribusi data sehingga nilai berada antara 0 dan 1. Formula: `x' = (x - x_min) / (x_max - x_min)`

**Kapan menggunakan masing-masing:**
- `StandardScaler()`: Ketika data mengikuti distribusi Gaussian (normal). Berguna untuk regresi linear, SVM.
- `MinMaxScaler()`: Ketika ingin mempertahankan hubungan antar nilai sambil memastikan semua fitur berkontribusi sama.
- `Normalizer()`: Khususnya berguna untuk data sparse atau ketika ingin memperlakukan setiap sampel secara setara terlepas dari besarnya.

> **Catatan penting:** Metode berbasis pohon seperti Decision Tree dan Random Forest bekerja menggunakan nilai data asli, sehingga scaling sebelum pemodelan akan berdampak negatif pada hasil.


In [6]:
from sklearn.preprocessing import StandardScaler

# Inisialisasi StandardScaler
scaler = StandardScaler()

# Fit dan transform menggunakan DataFrame hasil IterativeImputer
scaled_data = scaler.fit_transform(iterative_imputed_df)
scaled_df = pd.DataFrame(scaled_data, columns=iterative_imputed_df.columns)

print("Dataset setelah StandardScaler:")
print(f"Mean setiap kolom (seharusnya mendekati 0): {scaled_df.mean().round(4).values}")
print(f"Std setiap kolom (seharusnya mendekati 1): {scaled_df.std().round(4).values}")
display(scaled_df.head())


Dataset setelah StandardScaler:
Mean setiap kolom (seharusnya mendekati 0): [-0.  0.  0. -0.  0. -0. -0. -0.  0. -0.]
Std setiap kolom (seharusnya mendekati 1): [1.026 1.026 1.026 1.026 1.026 1.026 1.026 1.026 1.026 1.026]


,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.2719,0.0195,-0.3731,-0.8943,0.1888,0.0318,0.0019,1.6838,-0.7809,-0.0036
1,0.7560,-1.8749,-1.5922,-1.0439,-0.3917,0.0859,1.4269,-0.0005,0.6525,1.4291
2,-0.0009,1.7835,0.2313,1.3120,0.0029,1.0216,0.0009,0.8397,-2.4099,-1.4723
3,0.0030,-1.2145,1.0568,1.4580,-1.4118,1.5313,1.5513,-0.0018,-0.3879,1.0532
4,-1.3971,-0.0298,1.2459,0.5599,0.6085,0.7926,0.4518,-0.0013,0.2105,1.6863


In [7]:
from sklearn.preprocessing import MinMaxScaler

# Inisialisasi MinMaxScaler
minmax_scaler = MinMaxScaler()

# Fit dan transform
minmax_scaled_data = minmax_scaler.fit_transform(iterative_imputed_df)
minmax_scaled_df = pd.DataFrame(minmax_scaled_data, columns=iterative_imputed_df.columns)

print("Dataset setelah MinMaxScaler:")
print(f"Nilai minimum setiap kolom (seharusnya 0): {minmax_scaled_df.min().round(4).values}")
print(f"Nilai maksimum setiap kolom (seharusnya 1): {minmax_scaled_df.max().round(4).values}")
display(minmax_scaled_df.head())


Dataset setelah MinMaxScaler:
Nilai minimum setiap kolom (seharusnya 0): [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Nilai maksimum setiap kolom (seharusnya 1): [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.6035,0.5178,0.3805,0.3546,0.5690,0.5559,0.4350,0.9628,0.4022,0.4650
1,0.7214,0.0000,0.0004,0.3136,0.4131,0.5719,0.8331,0.5386,0.7560,0.9186
2,0.5371,1.0000,0.5690,0.9599,0.5191,0.8490,0.4347,0.7502,0.0000,0.0000
3,0.5380,0.1805,0.8264,1.0000,0.1390,1.0000,0.8678,0.5383,0.4992,0.7996
4,0.1972,0.5043,0.8854,0.7536,0.6818,0.7812,0.5607,0.5384,0.6469,1.0000


In [8]:
from sklearn.preprocessing import Normalizer

# Inisialisasi Normalizer (L2 normalization secara default)
normalizer = Normalizer()

# Fit dan transform
normalized_data = normalizer.fit_transform(iterative_imputed_df)
normalized_df = pd.DataFrame(normalized_data, columns=iterative_imputed_df.columns)

print("Dataset setelah Normalizer (L2):")
display(normalized_df.head())

# Verifikasi: L2 norm setiap baris seharusnya = 1
row_norms = np.linalg.norm(normalized_df.values, axis=1)
print(f"L2 norm setiap baris (seharusnya mendekati 1.0): {row_norms[:5].round(4)}")


Dataset setelah Normalizer (L2):


,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.3392,0.3136,0.2423,0.2000,0.2882,0.3385,0.2952,0.5189,0.2374,0.2786
1,0.3767,0.0515,0.0347,0.1686,0.2046,0.3237,0.4419,0.2732,0.4084,0.4924
2,0.2643,0.4834,0.3001,0.4262,0.2320,0.4232,0.2574,0.3536,0.0089,0.0207
3,0.2438,0.1166,0.3878,0.4077,0.0782,0.4502,0.3923,0.2347,0.2346,0.3709
4,0.0959,0.2489,0.4175,0.3165,0.2743,0.3659,0.2849,0.2371,0.3046,0.4637


L2 norm setiap baris (seharusnya mendekati 1.0): [1. 1. 1. 1. 1.]


---
## 4. Encoding Variabel Kategorikal

### Penjelasan Teori

Variabel kategorikal adalah fitur yang merepresentasikan nilai diskret seperti kategori, label, atau grup. Sebagian besar algoritma ML memerlukan input numerik, sehingga konversi variabel kategorikal ke format numerik adalah langkah yang esensial.

**Dua jenis variabel kategorikal:**

1. **Variabel Nominal:** Merepresentasikan kategori tanpa urutan intrinsik (misalnya, warna, merek, departemen). Gunakan `OneHotEncoder()`.

2. **Variabel Ordinal:** Memiliki urutan yang jelas antar kategori (misalnya, rating 1-5, tingkat jabatan: junior < senior < manager). Dapat menggunakan `LabelEncoder()` atau `OrdinalEncoder()`.

**Pilihan encoding:**

- `OneHotEncoder()`: Membuat kolom biner baru untuk setiap kategori. Cocok untuk variabel nominal. **Kelemahan:** Dapat menghasilkan dataset yang sangat besar dan sparse jika terlalu banyak kategori (curse of dimensionality).

- `LabelEncoder()`: Memberikan integer unik ke setiap kategori. **Perhatian:** Dapat memperkenalkan hubungan ordinal yang tidak diinginkan, sehingga lebih tepat untuk variabel ordinal atau label target.

- `ColumnTransformer()`: Memungkinkan penerapan teknik preprocessing yang berbeda pada kolom yang berbeda secara serentak, sangat berguna untuk dataset campuran (numerik dan kategorikal).


In [9]:
# Membuat dataset toy dengan variabel kategorikal
np.random.seed(2024)

categories = ["A", "B", "C", "D"]
categorical_data = pd.DataFrame({
    "Department": np.random.choice(categories, size=20),
    "Position": np.random.choice(["Junior", "Senior", "Manager"], size=20),
    "Location": np.random.choice(["NY", "SF", "LA", "CHI"], size=20),
})

print("Dataset kategorikal (5 baris pertama):")
display(categorical_data.head())
print(f"Shape: {categorical_data.shape}")


Dataset kategorikal (5 baris pertama):


,Department,Position,Location
0,A,Manager,LA
1,C,Manager,LA
2,A,Junior,NY
3,A,Manager,LA
4,D,Manager,NY


Shape: (20, 3)


### 4.1 OneHotEncoder()

One-hot encoding membuat kolom biner untuk setiap kategori yang unik. Jika fitur `Department` memiliki 4 kategori (A, B, C, D), one-hot encoding akan membuat 4 fitur biner baru. Setiap record akan memiliki nilai 1 pada kolom yang sesuai dengan kategorinya dan 0 pada yang lain.


In [10]:
from sklearn.preprocessing import OneHotEncoder

# Inisialisasi OneHotEncoder
onehot_encoder = OneHotEncoder(sparse_output=False)

# Fit dan transform
onehot_encoded_data = onehot_encoder.fit_transform(categorical_data)
onehot_encoded_df = pd.DataFrame(
    onehot_encoded_data,
    columns=onehot_encoder.get_feature_names_out()
)

print(f"Shape sebelum encoding: {categorical_data.shape}")
print(f"Shape setelah OneHotEncoder: {onehot_encoded_df.shape}")
print()
print("Hasil OneHotEncoder (5 baris pertama):")
display(onehot_encoded_df.head())


Shape sebelum encoding: (20, 3)
Shape setelah OneHotEncoder: (20, 11)

Hasil OneHotEncoder (5 baris pertama):


,Department_A,Department_B,Department_C,Department_D,Position_Junior,Position_Manager,Position_Senior,Location_CHI,Location_LA,Location_NY,Location_SF
0,1.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,1.0000,0.0000,0.0000
1,0.0000,0.0000,1.0000,0.0000,0.0000,1.0000,0.0000,0.0000,1.0000,0.0000,0.0000
2,1.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000
3,1.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,1.0000,0.0000,0.0000
4,0.0000,0.0000,0.0000,1.0000,0.0000,1.0000,0.0000,0.0000,0.0000,1.0000,0.0000


### 4.2 LabelEncoder()

Label encoding memberikan integer unik untuk setiap kategori dalam fitur nominal. Meskipun sederhana dan efisien, metode ini dapat memperkenalkan hubungan ordinal yang tidak diinginkan antara kategori.

**Contoh masalah:** Jika warna diencode sebagai Merah=0, Hijau=1, Biru=2, algoritma mungkin menginterpretasikan ini seolah Biru "lebih besar" dari Merah, yang tidak benar.


In [11]:
from sklearn.preprocessing import LabelEncoder

# Inisialisasi LabelEncoder
label_encoder = LabelEncoder()

# Membuat DataFrame baru untuk menyimpan nilai yang diencoding
label_encoded_df = pd.DataFrame()

# Fit dan transform setiap kolom kategorikal
for column in categorical_data.columns:
    label_encoded_df[f"{column}_encoded"] = (
        label_encoder.fit_transform(categorical_data[column])
    )

print("Hasil LabelEncoder (5 baris pertama):")
display(label_encoded_df.head())

# Menampilkan mapping yang digunakan
print()
print("Contoh mapping untuk kolom 'Position':")
label_encoder.fit(categorical_data["Position"])
for idx, cls in enumerate(label_encoder.classes_):
    print(f"  {cls} -> {idx}")


Hasil LabelEncoder (5 baris pertama):


,Department_encoded,Position_encoded,Location_encoded
0,0,1,1
1,2,1,1
2,0,0,2
3,0,1,1
4,3,1,2



Contoh mapping untuk kolom 'Position':
  Junior -> 0
  Manager -> 1
  Senior -> 2


### 4.3 ColumnTransformer()

`ColumnTransformer()` memungkinkan penerapan teknik preprocessing yang berbeda pada kolom-kolom yang berbeda secara serentak. Ini sangat berguna ketika bekerja dengan dataset yang memiliki campuran fitur numerik dan kategorikal.


In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Dataset campuran (numerik + kategorikal)
np.random.seed(2024)
mixed_data = pd.DataFrame({
    "Age": np.random.randint(25, 65, size=20),
    "Salary": np.round(np.random.normal(60000, 15000, size=20), 2),
    "Experience": np.random.randint(1, 20, size=20),
    "Department": np.random.choice(["IT", "HR", "Sales", "Finance"], size=20),
    "Position": np.random.choice(["Junior", "Senior", "Manager"], size=20),
})

print("Dataset campuran (5 baris pertama):")
display(mixed_data.head())


Dataset campuran (5 baris pertama):


,Age,Salary,Experience,Department,Position
0,33,59420.3600,12,Finance,Manager
1,57,82895.9200,17,Sales,Junior
2,25,38165.7600,16,Finance,Manager
3,52,38242.3600,7,IT,Junior
4,61,55088.6500,8,Sales,Manager


In [13]:
# Definisikan ColumnTransformer
column_transformer = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), ["Age", "Salary", "Experience"]),
        ("cat", OneHotEncoder(sparse_output=False), ["Department", "Position"]),
    ],
    remainder="passthrough",
)

# Fit dan transform
transformed_data = column_transformer.fit_transform(mixed_data)

# Buat nama kolom yang deskriptif
numeric_cols = ["Age_scaled", "Salary_scaled", "Experience_scaled"]
categorical_cols = list(
    column_transformer
    .named_transformers_["cat"]
    .get_feature_names_out(["Department", "Position"])
)

# Buat DataFrame hasil transformasi
transformed_df = pd.DataFrame(
    transformed_data,
    columns=numeric_cols + categorical_cols
)

print(f"Shape setelah ColumnTransformer: {transformed_df.shape}")
print("Hasil ColumnTransformer (5 baris pertama):")
display(transformed_df.head())


Shape setelah ColumnTransformer: (20, 10)
Hasil ColumnTransformer (5 baris pertama):


,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager,Position_Senior
0,-1.0453,-0.3050,0.3273,1.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000
1,0.7277,1.1051,1.2625,0.0000,0.0000,0.0000,1.0000,1.0000,0.0000,0.0000
2,-1.6364,-1.5818,1.0754,1.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000
3,0.3583,-1.5772,-0.6078,0.0000,0.0000,1.0000,0.0000,1.0000,0.0000,0.0000
4,1.0232,-0.5652,-0.4208,0.0000,0.0000,0.0000,1.0000,0.0000,1.0000,0.0000


---
## 5. Introduction to Pipelines dalam scikit-learn

### Penjelasan Teori

Kelas `Pipeline()` dalam scikit-learn menawarkan solusi kuat untuk menyederhanakan proses preprocessing dan pelatihan model. Dengan memungkinkan pengguna menghubungkan berbagai langkah preprocessing dan pelatihan model menjadi satu objek, pipeline meningkatkan efisiensi kode dan mengurangi kemungkinan kesalahan.

**Keuntungan utama Pipeline:**

1. **Sequential execution:** Setiap langkah dieksekusi sesuai urutan yang ditentukan.
2. **Code simplification:** Mengkondensasikan banyak baris kode menjadi satu objek.
3. **Consistency:** Memastikan transformasi yang sama diterapkan saat training dan prediksi.
4. **Easier hyperparameter tuning:** Terintegrasi dengan `GridSearchCV`.
5. **Modularity:** Mendorong komponen yang dapat digunakan kembali.

**Alasan krusial pemisahan data SEBELUM transformasi:**

- **Mencegah Data Leakage:** Data leakage terjadi ketika informasi dari test set mempengaruhi proses training. Dengan split sebelum preprocessing, transformasi hanya didasarkan pada data training.
- **Evaluasi model yang realistis:** Mensimulasikan skenario dunia nyata di mana data baru yang belum dilihat dipresentasikan ke model.
- **Konsistensi transformasi:** Parameter yang dipelajari dari training (seperti mean dan std untuk scaling) diterapkan pada test set menggunakan `transform()`, bukan di-refit.


In [14]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Pisahkan fitur dan target (asumsikan kolom terakhir adalah target)
X = transformed_df.iloc[:, :-1]  # Semua kolom kecuali terakhir
y = transformed_df.iloc[:, -1]   # Kolom terakhir sebagai target

# Pisahkan data SEBELUM menerapkan transformasi apapun (mencegah data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2024
)
print(f"Ukuran data training: {X_train.shape}")
print(f"Ukuran data testing: {X_test.shape}")


Ukuran data training: (16, 9)
Ukuran data testing: (4, 9)


In [15]:
# Membuat pipeline dengan dua langkah: imputasi + scaling
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),  # Tangani missing values
    ("scaler", StandardScaler()),                  # Scale fitur
])

# Fit dan transform data training
X_train_transformed = pipeline.fit_transform(X_train)

# Transform data testing (HANYA transform, tidak fit ulang)
X_test_transformed = pipeline.transform(X_test)

# Buat DataFrames dengan data yang sudah ditransformasi
X_train_transformed = pd.DataFrame(
    X_train_transformed,
    columns=X_train.columns,
    index=X_train.index
)
X_test_transformed = pd.DataFrame(
    X_test_transformed,
    columns=X_test.columns,
    index=X_test.index
)

print("Data training setelah pipeline (5 baris pertama):")
display(X_train_transformed.head())


Data training setelah pipeline (5 baris pertama):


,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager
13,0.8343,-0.8675,-1.5598,-0.5774,-0.4804,2.0817,-0.7746,-0.5774,-1.0000
12,-0.6892,1.2828,-0.1509,-0.5774,2.0817,-0.4804,-0.7746,-0.5774,1.0000
16,0.5441,-1.1734,-1.7610,-0.5774,-0.4804,2.0817,-0.7746,-0.5774,1.0000
5,-1.3421,0.8215,-0.1509,-0.5774,-0.4804,-0.4804,1.2910,-0.5774,-1.0000
17,1.4147,0.6111,0.6541,-0.5774,-0.4804,-0.4804,1.2910,-0.5774,1.0000


In [16]:
# Visualisasi struktur pipeline
from sklearn import set_config
set_config(display='diagram')
print("Struktur pipeline:")
pipeline


Struktur pipeline:


Pipeline(steps=[('imputer', SimpleImputer()), ('scaler', StandardScaler())])

---
## 6. Feature Engineering

### Penjelasan Teori

**Feature engineering** adalah istilah umum yang merujuk pada dua aktivitas utama:

1. **Feature extraction (pembuatan fitur baru):** Mentransformasi data yang ada menjadi variabel baru yang mungkin menangkap pola atau hubungan penting. Contoh: menurunkan fitur `total_spending` dari fitur `price` dan `quantity`.

2. **Feature selection (seleksi fitur yang relevan):** Mengidentifikasi dan mempertahankan fitur yang paling informatif sambil membuang fitur yang tidak berkontribusi secara berarti pada kekuatan prediktif model.

Feature engineering yang efektif dapat menghasilkan model yang lebih sederhana yang melakukan generalisasi lebih baik pada data yang belum dilihat.

**Tiga pendekatan umum untuk feature selection:**

- **Filter methods:** Memilih fitur berdasarkan karakteristik statistik/matematika dari sebuah fitur (misalnya, korelasi dengan variabel target).
- **Wrapper methods:** Mengevaluasi subset fitur dengan membangun model dan mengevaluasi performa (misalnya, Recursive Feature Elimination/RFE).
- **Hybrid methods:** Menggabungkan elemen metode filter dan wrapper.

> **Prinsip penting dalam ML:** Model optimal adalah yang memberikan performa terbaik dengan jumlah fitur yang paling sedikit. Kesederhanaan selalu lebih disukai daripada kompleksitas.


### 6.1 PolynomialFeatures()

`PolynomialFeatures()` menghasilkan fitur polinomial dan interaksi dari fitur numerik yang sudah ada. Untuk degree=2, ini berarti menghasilkan fitur seperti x^2, x1*x2, dll. Teknik ini memungkinkan model linear untuk memodelkan hubungan non-linear.


In [17]:
from sklearn.preprocessing import PolynomialFeatures

# Untuk demonstrasi, gunakan sebagian kecil X_train_transformed
X_demo = X_train_transformed.iloc[:, :3]  # Ambil 3 fitur pertama untuk kejelasan

# Inisialisasi PolynomialFeatures dengan degree=2
poly = PolynomialFeatures(degree=2, include_bias=False)

# Fit dan transform
poly_features = poly.fit_transform(X_demo)
poly_features_df = pd.DataFrame(
    poly_features,
    columns=poly.get_feature_names_out(X_demo.columns)
)

print(f"Shape sebelum PolynomialFeatures: {X_demo.shape}")
print(f"Shape setelah PolynomialFeatures (degree=2): {poly_features_df.shape}")
print()
print("Nama fitur yang dihasilkan:")
print(poly.get_feature_names_out(X_demo.columns).tolist())


Shape sebelum PolynomialFeatures: (16, 3)
Shape setelah PolynomialFeatures (degree=2): (16, 9)

Nama fitur yang dihasilkan:
['Age_scaled', 'Salary_scaled', 'Experience_scaled', 'Age_scaled^2', 'Age_scaled Salary_scaled', 'Age_scaled Experience_scaled', 'Salary_scaled^2', 'Salary_scaled Experience_scaled', 'Experience_scaled^2']


### 6.2 KBinsDiscretizer()

`KBinsDiscretizer()` mengkonversi variabel kontinu menjadi variabel kategorikal dengan membaginya ke dalam interval diskret (bin). Berguna ketika hubungan antara fitur dan target bersifat non-linear dan lebih mudah ditangkap dengan cara diskret.

Parameter utama:
- `n_bins`: Jumlah bin (bucket) yang akan dibuat.
- `encode`: Cara mengkodekan output (`'ordinal'`, `'onehot'`, `'onehot-dense'`).
- `strategy`: Kriteria pembagian bin:
  - `'uniform'`: Semua bin memiliki lebar yang sama.
  - `'quantile'`: Setiap bin memiliki jumlah record yang sama.
  - `'kmeans'`: Bin dibuat berdasarkan K-means clustering.


In [18]:
from sklearn.preprocessing import KBinsDiscretizer

# Inisialisasi KBinsDiscretizer
kbins = KBinsDiscretizer(
    n_bins=3,
    encode="ordinal",
    strategy="uniform"
)

# Fit dan transform menggunakan X_train_transformed
binned_data = kbins.fit_transform(X_train_transformed)
binned_df = pd.DataFrame(
    binned_data,
    columns=X_train_transformed.columns
)

print("Hasil KBinsDiscretizer (n_bins=3, strategy='uniform'):")
display(binned_df.head())

# Menampilkan distribusi nilai di setiap bin
print()
print("Distribusi nilai pada kolom pertama:")
print(binned_df.iloc[:, 0].value_counts().sort_index())


Hasil KBinsDiscretizer (n_bins=3, strategy='uniform'):


,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager
0,2.0000,0.0000,0.0000,0.0000,0.0000,2.0000,0.0000,0.0000,0.0000
1,0.0000,2.0000,1.0000,0.0000,2.0000,0.0000,0.0000,0.0000,2.0000
2,2.0000,0.0000,0.0000,0.0000,0.0000,2.0000,0.0000,0.0000,2.0000
3,0.0000,2.0000,1.0000,0.0000,0.0000,0.0000,2.0000,0.0000,0.0000
4,2.0000,1.0000,2.0000,0.0000,0.0000,0.0000,2.0000,0.0000,2.0000



Distribusi nilai pada kolom pertama:
Age_scaled
0.0000    8
2.0000    8
Name: count, dtype: int64


### 6.3 Recursive Feature Elimination (RFE)

`RFE()` adalah teknik wrapper yang secara rekursif menghilangkan fitur yang paling tidak penting berdasarkan peringkat importance dari estimator yang ditentukan. Proses ini diulangi sampai jumlah fitur yang diinginkan tercapai.

**Alur kerja RFE:**
1. Model dilatih menggunakan semua fitur.
2. Fitur yang paling tidak penting diidentifikasi dan dihilangkan.
3. Model dilatih ulang dengan fitur yang tersisa.
4. Proses diulangi hingga mencapai jumlah fitur yang ditentukan.

`RFECV()` menambahkan cross-validation ke pendekatan ini untuk optimasi lebih lanjut.


In [19]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

# Inisialisasi RFE dengan LinearRegression sebagai estimator
rfe = RFE(
    estimator=LinearRegression(),
    n_features_to_select=3  # Pilih 3 fitur terbaik
)

# Fit RFE
rfe.fit(X_train_transformed, y_train)

# Menampilkan peringkat fitur (1 = dipilih)
feature_ranking = pd.DataFrame({
    'Feature': X_train_transformed.columns,
    'Ranking': rfe.ranking_,
    'Selected': rfe.support_
}).sort_values('Ranking')

print("Peringkat fitur dari RFE:")
display(feature_ranking)
print()
print(f"Fitur terpilih: {X_train_transformed.columns[rfe.support_].tolist()}")


Peringkat fitur dari RFE:


,Feature,Ranking,Selected
1,Salary_scaled,1,True
7,Position_Junior,1,True
8,Position_Manager,1,True
5,Department_IT,2,False
0,Age_scaled,3,False
4,Department_HR,4,False
3,Department_Finance,5,False
2,Experience_scaled,6,False
6,Department_Sales,7,False



Fitur terpilih: ['Salary_scaled', 'Position_Junior', 'Position_Manager']


### 6.4 SelectFromModel()

`SelectFromModel()` memungkinkan pengguna memilih fitur berdasarkan bobot kepentingan yang berasal dari model tertentu. Metode ini sangat berguna ketika bekerja dengan model berbasis pohon seperti Random Forest atau gradient boosting, yang menyediakan informasi feature importance.

Parameter utama:
- `estimator`: Model yang digunakan untuk mengevaluasi kepentingan fitur.
- `threshold`: Ambang batas kepentingan fitur. Fitur dengan importance di bawah threshold akan dihilangkan. Dapat berupa float, atau string seperti `'mean'` atau `'median'`.


In [20]:
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LinearRegression

# Inisialisasi SelectFromModel
selector = SelectFromModel(
    estimator=LinearRegression(),
    prefit=False,
    threshold='mean'  # Gunakan mean importance sebagai threshold
)

# Fit selector
selector.fit(X_train_transformed, y_train)

# Dapatkan fitur yang dipilih
selected_features_mask = selector.get_support()
selected_features = X_train_transformed.columns[selected_features_mask].tolist()

# Tampilkan feature importance
feature_importance_df = pd.DataFrame({
    'Feature': X_train_transformed.columns,
    'Importance': np.abs(selector.estimator_.coef_),
    'Selected': selected_features_mask
}).sort_values('Importance', ascending=False)

print("Feature importances dan status seleksi:")
display(feature_importance_df)
print(f"Fitur terpilih: {selected_features}")


Feature importances dan status seleksi:


,Feature,Importance,Selected
8,Position_Manager,0.5000,True
7,Position_Junior,0.4330,True
2,Experience_scaled,0.0000,False
1,Salary_scaled,0.0000,False
0,Age_scaled,0.0000,False
3,Department_Finance,0.0000,False
4,Department_HR,0.0000,False
5,Department_IT,0.0000,False
6,Department_Sales,0.0000,False


Fitur terpilih: ['Position_Junior', 'Position_Manager']


---
## 7. Latihan Praktis: Comprehensive Pipeline dengan California Housing Dataset

### Deskripsi

Untuk mengkonsolidasikan semua yang telah dipelajari, kita akan membangun pipeline preprocessing komprehensif menggunakan dataset California Housing yang sudah disertakan dalam scikit-learn. Dataset ini berisi 20,640 record dan 8 fitur, di mana nilai target adalah harga rumah rata-rata per 100,000 rumah.

**Langkah-langkah:**
1. Load California Housing dataset.
2. Pisahkan data (train/test split).
3. Bangun pipeline komprehensif dengan minimal 3 langkah termasuk estimator sebagai langkah terakhir.
4. Fit pipeline ke dataset.
5. Evaluasi performa pipeline.


In [21]:
from sklearn.datasets import fetch_california_housing
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# 1. Load dataset California Housing
california = fetch_california_housing()
X_ca, y_ca = california.data, california.target
feature_names = california.feature_names

print(f"Jumlah sampel: {X_ca.shape[0]}")
print(f"Jumlah fitur: {X_ca.shape[1]}")
print(f"Nama fitur: {feature_names}")
print(f"Target: harga rumah median (dalam satuan $100,000)")


Jumlah sampel: 20640
Jumlah fitur: 8
Nama fitur: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
Target: harga rumah median (dalam satuan $100,000)


In [22]:
# 2. Pisahkan data sebelum transformasi apapun (mencegah data leakage)
X_train_ca, X_test_ca, y_train_ca, y_test_ca = train_test_split(
    X_ca, y_ca,
    test_size=0.2,
    random_state=42
)

print(f"Ukuran data training: {X_train_ca.shape}")
print(f"Ukuran data testing: {X_test_ca.shape}")


Ukuran data training: (16512, 8)
Ukuran data testing: (4128, 8)


In [23]:
# 3. Bangun pipeline komprehensif
comprehensive_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),  # Tangani missing values (jika ada)
    ("scaler", StandardScaler()),                    # Standardisasi fitur
    ("model", Ridge(alpha=1.0))                      # Model Ridge Regression
])

# 4. Fit pipeline ke data training
comprehensive_pipeline.fit(X_train_ca, y_train_ca)

# 5. Evaluasi pipeline
y_pred_ca = comprehensive_pipeline.predict(X_test_ca)

mse = mean_squared_error(y_test_ca, y_pred_ca)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_ca, y_pred_ca)

print("Evaluasi Pipeline Komprehensif pada Test Set:")
print(f"  Mean Squared Error (MSE): {mse:.4f}")
print(f"  Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"  R-squared (R2): {r2:.4f}")


Evaluasi Pipeline Komprehensif pada Test Set:
  Mean Squared Error (MSE): 0.5559
  Root Mean Squared Error (RMSE): 0.7456
  R-squared (R2): 0.5758


In [24]:
# Cross-validation untuk evaluasi yang lebih andal
cv_scores = cross_val_score(
    comprehensive_pipeline,
    X_train_ca, y_train_ca,
    cv=5,
    scoring='r2'
)

print("Hasil Cross-Validation (5-fold) pada data training:")
print(f"  CV R2 scores: {cv_scores.round(4)}")
print(f"  Mean CV R2: {cv_scores.mean():.4f}")
print(f"  Std CV R2: {cv_scores.std():.4f}")


Hasil Cross-Validation (5-fold) pada data training:
  CV R2 scores: [0.6201 0.613  0.6134 0.6107 0.6002]
  Mean CV R2: 0.6115
  Std CV R2: 0.0065


---
## Ringkasan Chapter 2

| Teknik | Kelas scikit-learn | Kegunaan |
|--------|-------------------|----------|
| Imputasi mean/median/modus | `SimpleImputer()` | Missing data ringan (<5%) |
| Imputasi berbasis KNN | `KNNImputer()` | Missing data sedang (5-10%), fitur numerik |
| Imputasi iteratif | `IterativeImputer()` | Missing data kompleks, semua tipe fitur |
| Standardisasi Z-score | `StandardScaler()` | Data Gaussian, model linear, SVM |
| Normalisasi min-max | `MinMaxScaler()` | Menjaga hubungan antar nilai, range [0,1] |
| Normalisasi L1/L2 | `Normalizer()` | Data sparse, equalizing sample magnitude |
| One-hot encoding | `OneHotEncoder()` | Variabel nominal |
| Label encoding | `LabelEncoder()` | Variabel ordinal atau label target |
| Transformasi campuran | `ColumnTransformer()` | Dataset dengan tipe fitur berbeda |
| Otomatisasi workflow | `Pipeline()` | Menghubungkan semua langkah preprocessing |
| Fitur polinomial | `PolynomialFeatures()` | Menangkap hubungan non-linear |
| Diskretisasi | `KBinsDiscretizer()` | Variabel kontinu ke kategorikal |
| Eliminasi fitur rekursif | `RFE()` | Seleksi fitur berbasis wrapper |
| Seleksi berbasis model | `SelectFromModel()` | Seleksi fitur berbasis importance score |

### Poin Kunci untuk Diingat:

1. **"Garbage in, garbage out"**: Kualitas data adalah fondasi dari semua model ML yang baik.
2. Selalu **split data sebelum transformasi** untuk mencegah data leakage.
3. Gunakan **`fit_transform()`** pada data training, tapi hanya **`transform()`** pada data testing.
4. `Pipeline()` adalah alat yang sangat efektif untuk menggabungkan semua langkah preprocessing dan pemodelan.
5. Tidak semua dataset memerlukan scaling (tree-based methods tidak memerlukannya) atau imputasi (beberapa algoritma dapat menangani missing values secara langsung).
6. Pilihan teknik preprocessing bergantung pada jenis data, distribusi, dan algoritma yang akan digunakan.
